# 04 - 手写玩具 RolloutStorage

本节目标: 你能自己写一个最小 on-policy buffer, 理解 `[T, N] -> [T*N]` flatten 和 mini-batch。

In [ ]:
import torch

T = 4
N = 3
obs_dim = 5
action_dim = 2

observations = torch.zeros(T, N, obs_dim)
actions = torch.zeros(T, N, action_dim)
rewards = torch.zeros(T, N, 1)
values = torch.zeros(T, N, 1)
old_log_prob = torch.zeros(T, N, 1)

print(observations.shape, actions.shape, rewards.shape)

## 写入 transition

真实 rsl_rl 的 `RolloutStorage.add_transition` 做的事情就是把当前 step 的数据 copy 到 `[step]` 位置。

In [ ]:
for step in range(T):
    observations[step] = torch.randn(N, obs_dim)
    actions[step] = torch.randn(N, action_dim)
    rewards[step] = torch.randn(N, 1)
    values[step] = torch.randn(N, 1)
    old_log_prob[step] = torch.randn(N, 1)

print('one step obs:', observations[0].shape)
print('all obs:', observations.shape)

## flatten 成 batch

PPO update 不再关心时间和 env 两个维度, 它把 `[T, N]` 展平为 `[T*N]`。

In [ ]:
flat_obs = observations.flatten(0, 1)
flat_actions = actions.flatten(0, 1)
flat_rewards = rewards.flatten(0, 1)

print('flat_obs:', flat_obs.shape)
print('flat_actions:', flat_actions.shape)
print('flat_rewards:', flat_rewards.shape)

## mini-batch 采样

真实代码用 `torch.randperm` 打乱索引。

In [ ]:
batch_size = T * N
num_mini_batches = 2
mini_batch_size = batch_size // num_mini_batches
indices = torch.randperm(batch_size)

for i in range(num_mini_batches):
    batch_idx = indices[i * mini_batch_size:(i + 1) * mini_batch_size]
    print('mini batch', i, 'idx:', batch_idx.tolist(), 'obs:', flat_obs[batch_idx].shape)

## 作业

1. 把 `T=8, N=4`, 预测 flat batch size。
2. 增加 `returns` 和 `advantages` 两个 buffer。
3. 写一个函数 `yield_minibatches(tensor, num_mini_batches)`。